In [1]:
BUILD_KIND = "asr"

MAPPING_HINT = "aic-mapping"
CODE_HINT = "update-script"
RAW_HINT = "asr"       # sửa theo tên Dataset ASR thực tế
CONFIG_HINT = None

In [2]:
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")

dataset_roots = []

for category in ("datasets", "models", "notebooks"):
    category_root = INPUT_ROOT / category

    if not category_root.exists():
        continue

    for owner_root in category_root.iterdir():
        if not owner_root.is_dir():
            continue

        for dataset_root in owner_root.iterdir():
            if dataset_root.is_dir():
                dataset_roots.append(dataset_root)

print("Các input đang được gắn:")

for root in dataset_roots:
    print("-", root)

Các input đang được gắn:
- /kaggle/input/datasets/blonton/asr-part1
- /kaggle/input/datasets/blonton/aic-mapping
- /kaggle/input/models/blonton/update-script-v2


In [3]:
from pathlib import Path


def find_input_root(hint, required=True):
    if not hint:
        return None

    hint = hint.casefold()

    matches = [
        path
        for path in dataset_roots
        if hint in path.name.casefold()
        or hint in str(path).casefold()
    ]

    if not matches:
        if required:
            raise FileNotFoundError(
                f"Không tìm thấy Kaggle input chứa tên: {hint}"
            )
        return None

    if len(matches) > 1:
        print(f"Có nhiều kết quả cho {hint!r}:")
        for path in matches:
            print(" -", path)

    return matches[0]


MAPPING_SOURCE_ROOT = find_input_root(MAPPING_HINT)
CODE_SOURCE_ROOT = find_input_root(CODE_HINT)
RAW_SOURCE_ROOT = find_input_root(RAW_HINT)

CONFIG_SOURCE_ROOT = find_input_root(
    CONFIG_HINT,
    required=False,
)

print("MAPPING:", MAPPING_SOURCE_ROOT)
print("CODE   :", CODE_SOURCE_ROOT)
print("RAW    :", RAW_SOURCE_ROOT)
print("CONFIG :", CONFIG_SOURCE_ROOT)

MAPPING: /kaggle/input/datasets/blonton/aic-mapping
CODE   : /kaggle/input/models/blonton/update-script-v2
RAW    : /kaggle/input/datasets/blonton/asr-part1
CONFIG : None


In [4]:
import os
from pathlib import Path

required_build_script = {
    "object": "build_object_index.py",
    "ocr": "build_ocr_index.py",
    "asr": "build_asr_index.py",
}[BUILD_KIND]


def find_script_candidates(filename):
    candidates = []

    # Chỉ dò các input có khả năng chứa code, tránh quét bộ keyframe lớn
    search_roots = []

    for category in ("models", "datasets"):
        category_root = Path("/kaggle/input") / category

        if not category_root.exists():
            continue

        for owner_root in category_root.iterdir():
            if not owner_root.is_dir():
                continue

            for input_root in owner_root.iterdir():
                if not input_root.is_dir():
                    continue

                name = input_root.name.casefold()

                if (
                    "script" in name
                    or "search" in name
                    or "code" in name
                    or CODE_HINT.casefold() in name
                ):
                    search_roots.append(input_root)

    print("Các nguồn code sẽ kiểm tra:")

    for root in search_roots:
        print("-", root)

    for root in search_roots:
        root_depth = len(root.parts)

        for current, directories, files in os.walk(root):
            current_path = Path(current)
            depth = len(current_path.parts) - root_depth

            # Đủ sâu cho cấu trúc Kaggle Model:
            # model/framework/variation/version/scripts
            if depth >= 12:
                directories[:] = []

            if filename in files:
                candidates.append(current_path / filename)

    return candidates


script_candidates = find_script_candidates(
    required_build_script
)

print(f"\nKết quả tìm {required_build_script}:")

for path in script_candidates:
    print("-", path)

assert script_candidates, (
    f"Không tìm thấy {required_build_script}. "
    "Model code đang gắn có thể là version cũ hoặc thiếu file này."
)

# Ưu tiên source có nhiều file pipeline cần thiết nhất
required_companion_files = {
    "search_types.py",
    "stdio_setup.py",
    "object_retriever.py",
}

def source_score(script_path):
    directory = script_path.parent
    existing = {
        path.name
        for path in directory.glob("*.py")
    }
    return len(existing & required_companion_files)


BUILD_SCRIPT_SOURCE = max(
    script_candidates,
    key=source_score,
)

CODE_SCRIPTS_SOURCE = BUILD_SCRIPT_SOURCE.parent

MAPPING_SOURCE = None

for candidate in [
    MAPPING_SOURCE_ROOT / "artifacts" / "clip_row_mapping.jsonl",
    MAPPING_SOURCE_ROOT / "clip_row_mapping.jsonl",
]:
    if candidate.is_file():
        MAPPING_SOURCE = candidate
        break

if MAPPING_SOURCE is None:
    for current, directories, files in os.walk(MAPPING_SOURCE_ROOT):
        if "clip_row_mapping.jsonl" in files:
            MAPPING_SOURCE = (
                Path(current) / "clip_row_mapping.jsonl"
            )
            break

assert MAPPING_SOURCE, (
    "Không tìm thấy clip_row_mapping.jsonl trong aic-mapping"
)

print("\n✅ Build script:", BUILD_SCRIPT_SOURCE)
print("✅ Thư mục code:", CODE_SCRIPTS_SOURCE)
print("✅ Mapping:", MAPPING_SOURCE)

Các nguồn code sẽ kiểm tra:
- /kaggle/input/models/blonton/update-script-v2

Kết quả tìm build_asr_index.py:
- /kaggle/input/models/blonton/update-script-v2/pytorch/default/1/build_asr_index.py

✅ Build script: /kaggle/input/models/blonton/update-script-v2/pytorch/default/1/build_asr_index.py
✅ Thư mục code: /kaggle/input/models/blonton/update-script-v2/pytorch/default/1
✅ Mapping: /kaggle/input/datasets/blonton/aic-mapping/artifacts/clip_row_mapping.jsonl


In [5]:
import shutil
from pathlib import Path

PROJECT_ROOT = Path("/kaggle/working/aic_practice")
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
DATA_DIR = PROJECT_ROOT / "data"
CONFIG_DIR = PROJECT_ROOT / "config"

for directory in (
    PROJECT_ROOT,
    SCRIPTS_DIR,
    ARTIFACTS_DIR,
    DATA_DIR,
    CONFIG_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_ROOT)

Project: /kaggle/working/aic_practice


In [6]:
copied_files = []

for source in CODE_SCRIPTS_SOURCE.glob("*.py"):
    destination = SCRIPTS_DIR / source.name
    shutil.copy2(source, destination)
    copied_files.append(source.name)

assert required_build_script in copied_files, (
    f"Chưa copy được {required_build_script}"
)

print("Số file code đã copy:", len(copied_files))
print("Build script:", SCRIPTS_DIR / required_build_script)

Số file code đã copy: 33
Build script: /kaggle/working/aic_practice/scripts/build_asr_index.py


In [7]:
MAPPING_TARGET = ARTIFACTS_DIR / "clip_row_mapping.jsonl"

shutil.copy2(
    MAPPING_SOURCE,
    MAPPING_TARGET,
)

print("Mapping:", MAPPING_TARGET)
print(
    "Dung lượng:",
    round(MAPPING_TARGET.stat().st_size / 1024**2, 2),
    "MB",
)

Mapping: /kaggle/working/aic_practice/artifacts/clip_row_mapping.jsonl
Dung lượng: 109.47 MB


In [8]:
DATA_TARGET = DATA_DIR / {
    "object": "objects",
    "ocr": "ocr",
    "asr": "asr",
}[BUILD_KIND]

if DATA_TARGET.is_symlink():
    DATA_TARGET.unlink()
elif DATA_TARGET.exists():
    if DATA_TARGET.is_dir() and not any(DATA_TARGET.iterdir()):
        DATA_TARGET.rmdir()
    else:
        raise RuntimeError(
            f"{DATA_TARGET} đã tồn tại và không rỗng. "
            "Hãy kiểm tra trước khi thay thế."
        )

DATA_TARGET.symlink_to(
    RAW_SOURCE_ROOT,
    target_is_directory=True,
)

print("Nguồn:", RAW_SOURCE_ROOT)
print("Đã gắn vào:", DATA_TARGET)
print("Symlink:", DATA_TARGET.is_symlink())

Nguồn: /kaggle/input/datasets/blonton/asr-part1
Đã gắn vào: /kaggle/working/aic_practice/data/asr
Symlink: True


In [9]:
print("PROJECT_ROOT:", PROJECT_ROOT)
print("BUILD_KIND:", BUILD_KIND)
print("Build script tồn tại:", (SCRIPTS_DIR / required_build_script).is_file())
print("Mapping tồn tại:", MAPPING_TARGET.is_file())
print("Data target tồn tại:", DATA_TARGET.exists())
print("Data target là symlink:", DATA_TARGET.is_symlink())

sample_files = []

for suffix in ("*.json", "*.jsonl"):
    for path in DATA_TARGET.rglob(suffix):
        sample_files.append(path)

        if len(sample_files) >= 10:
            break

    if len(sample_files) >= 10:
        break

print("\nFile dữ liệu mẫu:")

for path in sample_files:
    print("-", path)

assert sample_files, (
    f"Không tìm thấy JSON/JSONL trong {DATA_TARGET}"
)

PROJECT_ROOT: /kaggle/working/aic_practice
BUILD_KIND: asr
Build script tồn tại: True
Mapping tồn tại: True
Data target tồn tại: True
Data target là symlink: True

File dữ liệu mẫu:
- /kaggle/working/aic_practice/data/asr/json_results/Videos_L22_a/L22_V024/L22_V024.json
- /kaggle/working/aic_practice/data/asr/json_results/Videos_L22_a/L22_V010/L22_V010.json
- /kaggle/working/aic_practice/data/asr/json_results/Videos_L22_a/L22_V018/L22_V018.json
- /kaggle/working/aic_practice/data/asr/json_results/Videos_L22_a/L22_V022/L22_V022.json
- /kaggle/working/aic_practice/data/asr/json_results/Videos_L22_a/L22_V013/L22_V013.json
- /kaggle/working/aic_practice/data/asr/json_results/Videos_L22_a/L22_V021/L22_V021.json
- /kaggle/working/aic_practice/data/asr/json_results/Videos_L22_a/L22_V006/L22_V006.json
- /kaggle/working/aic_practice/data/asr/json_results/Videos_L22_a/L22_V025/L22_V025.json
- /kaggle/working/aic_practice/data/asr/json_results/Videos_L22_a/L22_V017/L22_V017.json
- /kaggle/working

In [10]:
%cd /kaggle/working/aic_practice
!python -u scripts/build_asr_index.py \
    --input data/asr \
    --output artifacts/asr_index.sqlite3

/kaggle/working/aic_practice
Đã index 10,000 ASR segment...
Tạo ASR index thành công: artifacts/asr_index.sqlite3
  Segments: 11,061; rỗng: 12; sai video_id: 0


In [11]:
from pathlib import Path

expected_index = {
    "object": PROJECT_ROOT / "artifacts/object_index.sqlite3",
    "ocr": PROJECT_ROOT / "artifacts/ocr_index.sqlite3",
    "asr": PROJECT_ROOT / "artifacts/asr_index.sqlite3",
}[BUILD_KIND]

assert expected_index.is_file(), (
    f"Build chưa tạo được {expected_index}"
)

print("✅ Build thành công:", expected_index)
print(
    "Dung lượng:",
    round(expected_index.stat().st_size / 1024**2, 2),
    "MB",
)

✅ Build thành công: /kaggle/working/aic_practice/artifacts/asr_index.sqlite3
Dung lượng: 7.62 MB


In [12]:
import shutil
from pathlib import Path

EXPORT_DIR = Path(
    f"/kaggle/working/aic2026_{BUILD_KIND}_index"
)

EXPORT_ARTIFACTS = EXPORT_DIR / "artifacts"
EXPORT_ARTIFACTS.mkdir(
    parents=True,
    exist_ok=True,
)

shutil.copy2(
    expected_index,
    EXPORT_ARTIFACTS / expected_index.name,
)

shutil.copy2(
    MAPPING_TARGET,
    EXPORT_ARTIFACTS / "clip_row_mapping.jsonl",
)

archive_path = shutil.make_archive(
    str(EXPORT_DIR),
    "zip",
    root_dir=EXPORT_DIR,
)

print("✅ Thư mục output:", EXPORT_DIR)
print("✅ File ZIP:", archive_path)

✅ Thư mục output: /kaggle/working/aic2026_asr_index
✅ File ZIP: /kaggle/working/aic2026_asr_index.zip
